In [ ]:
# 1. GPU Kontrolü
import torch
print('GPU mevcut:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('UYARI: GPU bulunamadı, eğitim CPU ile çok yavaş olacak.')

In [ ]:
# 2. Bağımlılıkları Kur (transformers 4.44+ gerekli)
!pip install -q transformers==4.44.0 datasets==2.19.0 evaluate==0.4.1 accelerate==0.29.3 seaborn scikit-learn
print('Kurulum tamamlandı!')

In [ ]:
# 3. Google Drive Bağla (modelleri kalıcı kaydetmek için)
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/TurkceHaberAnaliz'
os.makedirs(PROJECT_DIR, exist_ok=True)
print('Proje dizini:', PROJECT_DIR)

In [ ]:
# 4. Çalışma dizinini ayarla ve dosyaları yükle
# utils.py, train_topic.py, train_sentiment.py dosyalarını /content/ altına yükleyin.
import os
os.chdir('/content')
print('Çalışma dizini:', os.getcwd())
!ls

In [ ]:
# 5. utils.py — Colab'a yaz 
utils_code = '''
import re, os, json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import evaluate
from sklearn.metrics import classification_report, confusion_matrix

def clean_text(text):
    if not isinstance(text, str): return ""
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"http\\S+|www\\.\\S+", " ", text)
    text = re.sub(r"\\S+@\\S+", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    return text

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1  = f1_metric.compute(predictions=predictions, references=labels, average="weighted")
    return {"accuracy": acc["accuracy"], "f1": f1["f1"]}

def plot_confusion_matrix(y_true, y_pred, labels, save_path=None):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(max(6,len(labels)), max(5,len(labels)-1)))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_xlabel("Tahmin"); ax.set_ylabel("Gercek"); ax.set_title("Confusion Matrix")
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
        plt.savefig(save_path, dpi=150)
    plt.show(); plt.close(fig)

def plot_training_history(trainer, save_path=None, title="Training History"):
    log = trainer.state.log_history
    tl, te, vl, ve = [], [], [], []
    for e in log:
        if "loss" in e and "eval_loss" not in e: tl.append(e["loss"]); te.append(e.get("epoch",len(tl)))
        if "eval_loss" in e: vl.append(e["eval_loss"]); ve.append(e.get("epoch",len(vl)))
    fig, ax = plt.subplots(figsize=(8,5))
    if tl: ax.plot(te, tl, label="Train Loss", marker="o", color="#2563EB")
    if vl: ax.plot(ve, vl, label="Val Loss",   marker="s", color="#DC2626")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.set_title(title); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
        plt.savefig(save_path, dpi=150)
    plt.show(); plt.close(fig)

def save_classification_report(y_true, y_pred, labels, save_path):
    rpt_str  = classification_report(y_true, y_pred, target_names=labels)
    rpt_dict = classification_report(y_true, y_pred, target_names=labels, output_dict=True)
    print(rpt_str)
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
    with open(save_path,"w",encoding="utf-8") as f: json.dump(rpt_dict, f, ensure_ascii=False, indent=2)
    return rpt_dict

def save_label_mapping(label2id, id2label, save_dir):
    os.makedirs(save_dir, exist_ok=True)
    with open(os.path.join(save_dir,"label_mapping.json"),"w",encoding="utf-8") as f:
        json.dump({"label2id":label2id,"id2label":id2label}, f, ensure_ascii=False, indent=2)

def load_label_mapping(save_dir):
    with open(os.path.join(save_dir,"label_mapping.json"),"r",encoding="utf-8") as f:
        m = json.load(f)
    return m["label2id"], {int(k):v for k,v in m["id2label"].items()}
'''
with open('utils.py','w',encoding='utf-8') as f:
    f.write(utils_code)
print('utils.py yazildi.')

In [ ]:
# 6. Sınıflandırma Eğitimi 
!python train_topic.py

In [ ]:
# 7. Duygu Analizi Eğitimi 
!python train_sentiment.py

In [ ]:
# 8. Sonuçları Görüntüle
import json, os
for task in ['topic','sentiment']:
    path = f'data/{task}/{task}_report.json'
    if os.path.exists(path):
        with open(path) as f: r = json.load(f)
        print(f'\n=== {task.upper()} ===')
        print(f'Accuracy : {r["accuracy"]:.4f}')
        print(f'Macro F1 : {r["macro avg"]["f1-score"]:.4f}')
    else:
        print(f'{path} bulunamadi, once egitimi calistirin.')

In [ ]:
# 9. Modelleri Drive'a Yedekle 
import shutil, os
for folder in ['models','data']:
    dst = os.path.join(PROJECT_DIR, folder)
    if os.path.exists(folder):
        shutil.copytree(folder, dst, dirs_exist_ok=True)
        print(f'{folder} -> Drive kopyalandi')
    else:
        print(f'{folder} henuz olusturulmamis, once egitimi calistirin.')

In [ ]:
# 10. Hizli Tahmin Testi 
import os
from transformers import pipeline


TOPIC_PATH = '/content/models/topic_model'
SENT_PATH  = '/content/models/sentiment_model'

if not os.path.isdir(TOPIC_PATH) or not os.path.isdir(SENT_PATH):
    missing = [p for p in [TOPIC_PATH, SENT_PATH] if not os.path.isdir(p)]
    print('HATA: Su klasorler bulunamadi:')
    for p in missing: print(' -', p)
    print('Lutfen once Hucre 6 (train_topic.py) ve Hucre 7 (train_sentiment.py) calistirin!')
else:
    topic_pipe = pipeline('text-classification', model=TOPIC_PATH)
    sent_pipe  = pipeline('text-classification', model=SENT_PATH)
    test_text = 'Borsa Istanbul bugun rekor kirdi, yatirimcilar memnun.'
    print('Metin:', test_text)
    print('Konu :', topic_pipe(test_text))
    print('Duygu:', sent_pipe(test_text))